In [2]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score
import scipy.stats as stats



In [3]:
def load_nc_data(year: int):
    nc_file_buoy = f'../data/years_dataframe/{year}_data.nc'
    ds_buoy = xr.open_dataset(nc_file_buoy)
    nc_file_wind = f'../data/years_dataframe/{year}_continous_wind_data.nc'
    ds_wind = xr.open_dataset(nc_file_wind)
    ds_wind_time_steps = ds_wind['time'].values
    ds_buoys_time_steps = ds_buoy['time'].values

    # Find common time steps
    common_time_steps = np.intersect1d(ds_wind_time_steps, ds_buoys_time_steps)


    ds_wind_filtered = ds_wind.sel(time=common_time_steps)
    ds_buoy_filtered = ds_buoy.sel(time=common_time_steps)
    
    return ds_buoy_filtered, ds_wind_filtered

In [5]:
def calculate_proximity(proximity_type: str, buoy_data: np.array, interpolated_data: np.array, ax_: plt.Axes, station: str):
    if proximity_type == 'correlation':
        value = np.corrcoef(buoy_data, buoy_data[:, 0])[0, 1]
        sns.regplot(x=buoy_data,y=interpolated_data , ax=ax_)
        ax_.set_title(f'Correlation for {station} with a correlation: {value:.2f}')
        
    elif proximity_type == 'time_series':
        ax_.plot(buoy_data, label='Buoy Data')
        ax_.plot(interpolated_data, label='Interpolated Data')
        ax_.set_title(f'Time Series for {station}')
        ax_.legend()
        value = None
    elif proximity_type == 'r2':
        value = r2_score(buoy_data, interpolated_data)
        sns.regplot(x=buoy_data, y=interpolated_data, ax=ax_)
        ax_.set_title(f'R2 Score for {station} with a score: {value:.2f}') 
    else:
        raise ValueError('Invalid proximity type')
    
    return value

In [6]:
def post_process_before_corr(ds_wind: xr.Dataset, ds_buoy_filtered: xr.Dataset, station: str, variable: str):
    
    if variable == 'u':
        buoy_data = np.array(ds_buoy_filtered.sel(station_id=station)['u_velocity'])
        wind_data = np.array(ds_wind.sel(station_id=station)['u_velocity'])
    elif variable == 'v':
        buoy_data = np.array(ds_buoy_filtered.sel(station_id=station)['v_velocity'])
        wind_data = np.array(ds_wind.sel(station_id=station)['v_velocity'])
    else:
        raise ValueError('Invalid correlation type')
    
    nan_idx = np.isnan(buoy_data)
    buoy_data = buoy_data[~nan_idx]
    wind_data = wind_data[~nan_idx]
    
    nan_idx = np.isnan(wind_data)
    buoy_data = buoy_data[~nan_idx]
    wind_data = wind_data[~nan_idx]
    
    return buoy_data, wind_data

In [9]:
year = 2019
ds_buoy_filtered, ds_wind_filtered = load_nc_data(year)
stations = ds_wind_filtered.station_id.values
proximity = 'r2'
variable = 'u'

In [ ]:
n_cols = 3
n_rows = (len(stations) // n_cols) + (len(stations) % n_cols > 0)

# Create a figure with subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten()  # Flatten the axes array for easy indexing  
similarity_values = []    
for i, station in enumerate(stations):    
    
    
    buoy_data, interp_data = post_process_before_corr(ds_wind_filtered, ds_buoy_filtered, station, variable)
    value_similarity = calculate_proximity(proximity, buoy_data, interp_data, axes[i], station)
    similarity_values.append(value_similarity)
    
    
    
# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()  # Adjust layout
plt.show()

## Gust Analysis

In [47]:
def plot_gust_analysis(plot_type: str, ax_: plt.Axes, ds_buoy_filtered: xr.Dataset, station: str):
    
    gust_speed = ds_buoy_filtered.sel(station_id=station)['GST']    
    wind_speed = np.sqrt(ds_buoy_filtered.sel(station_id=station)['u_velocity']**2 + ds_buoy_filtered.sel(station_id=station)['v_velocity']**2)
    time = ds_buoy_filtered.sel(station_id=station)['time']
    

    
    if plot_type == 'gust_factor':
        gust_factor = gust_speed / wind_speed
        sns.histplot(gust_factor, bins=30, kde=True, ax=ax_)
        ax_.set_title(f'Distribution of Gust Factor for {station}') 
        ax_.set_xlabel('Gust Factor')
        ax_.grid()
        
    elif plot_type == 'gev_distribution':
        # Fit a GEV distribution to the gust speed data
        nan_idx = np.isnan(gust_speed)
        gust_speed = gust_speed[~nan_idx]
        shape, loc, scale = stats.genextreme.fit(gust_speed)
        x = np.linspace(0, gust_speed.values.max(), 100)
        y = stats.genextreme.pdf(x, shape, loc, scale)

        # Plot the fitted GEV distribution
        ax_.hist(gust_speed, bins=30, density=True, alpha=0.6, color='g')
        ax_.plot(x, y, 'r-', lw=2)
        ax_.set_title(f'Extreme Value Distribution of Gust Speed for {station}')
        ax_.set_xlabel('Speed (m/s)')
        ax_.set_ylabel('Density')
        ax_.grid()
    elif plot_type == 'time_series':
        ax_.plot(time, wind_speed, label='Wind Speed (m/s)', color='blue')
        ax_.plot(time, gust_speed, label='Gust Speed (m/s)', color='red')
        ax_.set_title(f'Time Series of Wind Speed and Gust Speed for {station}')
        ax_.set_xlabel('Time')
        ax_.set_ylabel('Speed (m/s)')
        ax_.legend()
        ax_.grid() 
    elif plot_type == 'scatter':
        ax_.scatter(wind_speed, gust_speed)
        ax_.plot([0, 30], [0, 30], 'r--')
        ax_.set_title(f'Scatter Plot of Wind Speed and Gust Speed for {station}')
        ax_.set_xlabel('Wind Speed (m/s)')
        ax_.set_ylabel('Gust Speed (m/s)')
        ax_.grid()
        ax_.grid()
    elif plot_type == 'qq_plot':
        sns.boxplot(data=[wind_speed, gust_speed], palette='Set2', ax=ax_)
        ax_.set_title(f'Box Plot of Wind Speed and Gust Speed for {station}')
        ax_.set_ylabel('Speed (m/s)')
        ax_.grid()
    
    else:
        raise ValueError('Invalid proximity type')
    
    

In [ ]:
n_cols = 3
n_rows = (len(stations) // n_cols) + (len(stations) % n_cols > 0)

# Create a figure with subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten()  # Flatten the axes array for easy indexing  
similarity_values = []    
for i, station in enumerate(stations):  
    
    plot_gust_analysis('qq_plot', axes[i], ds_buoy_filtered, station)
    
# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()  # Adjust layout
plt.show()